# GRB preprocessing report

Inspect the cached Swift GRB light curves, compare short and long GRBs, and print one full example from each class.

In [75]:
from pathlib import Path

import numpy as np
import pandas as pd

from classes import GRBHDF5Dataset

PROJECT_ROOT = Path.cwd()
H5_FILE = PROJECT_ROOT / "data" / "processed" / "classipygrb" / "swift.hd5"
CLASS_NAMES = {0: "short", 1: "long"}

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

In [76]:
def class_label(label: int) -> str:
    return CLASS_NAMES.get(int(label), f"class_{label}")


def as_numpy(dataset: GRBHDF5Dataset) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = dataset.x.detach().cpu().numpy()
    labels = np.asarray(dataset.labels, dtype=int)
    t90 = np.asarray(dataset.t90, dtype=float)
    return x, labels, t90


def class_distribution(labels: np.ndarray) -> pd.DataFrame:
    rows = []
    total = labels.size
    for label in sorted(np.unique(labels)):
        count = int((labels == label).sum())
        rows.append(
            {
                "label": int(label),
                "class": class_label(int(label)),
                "count": count,
                "percent": 100.0 * count / total if total else 0.0,
            }
        )
    return pd.DataFrame(rows)


def describe_values(values: np.ndarray) -> dict[str, float]:
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "median": np.nan,
            "min": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "max": np.nan,
        }

    return {
        "mean": float(np.mean(finite)),
        "std": float(np.std(finite)),
        "median": float(np.median(finite)),
        "min": float(np.min(finite)),
        "q25": float(np.percentile(finite, 25)),
        "q75": float(np.percentile(finite, 75)),
        "max": float(np.max(finite)),
    }


def t90_summary(labels: np.ndarray, t90: np.ndarray) -> pd.DataFrame:
    rows = []
    for label in sorted(np.unique(labels)):
        stats = describe_values(t90[labels == label])
        rows.append({"class": class_label(int(label)), **stats})
    return pd.DataFrame(rows)

In [77]:
def overall_signal_summary(x: np.ndarray, labels: np.ndarray) -> pd.DataFrame:
    rows = []
    for label in sorted(np.unique(labels)):
        class_x = x[labels == label]
        rows.append(
            {
                "class": class_label(int(label)),
                "all_values_mean": float(np.mean(class_x)),
                "all_values_std": float(np.std(class_x)),
                "per_grb_mean": float(np.mean(class_x.mean(axis=(1, 2)))),
                "per_grb_peak_mean": float(np.mean(class_x.max(axis=(1, 2)))),
                "per_grb_sum_mean": float(np.mean(class_x.sum(axis=(1, 2)))),
                "zero_fraction": float(np.mean(class_x == 0.0)),
            }
        )
    return pd.DataFrame(rows)


def band_summary(x: np.ndarray, labels: np.ndarray, channel_columns: list[str]) -> pd.DataFrame:
    rows = []
    for label in sorted(np.unique(labels)):
        class_x = x[labels == label]
        for channel_idx, channel_name in enumerate(channel_columns):
            channel_x = class_x[:, :, channel_idx]
            per_grb_mean = channel_x.mean(axis=1)
            per_grb_peak = channel_x.max(axis=1)
            per_grb_sum = channel_x.sum(axis=1)
            rows.append(
                {
                    "class": class_label(int(label)),
                    "band": channel_name,
                    "timepoint_mean": float(np.mean(channel_x)),
                    "timepoint_std": float(np.std(channel_x)),
                    "per_grb_mean_avg": float(np.mean(per_grb_mean)),
                    "per_grb_peak_avg": float(np.mean(per_grb_peak)),
                    "per_grb_sum_avg": float(np.mean(per_grb_sum)),
                    "zero_fraction": float(np.mean(channel_x == 0.0)),
                }
            )
    return pd.DataFrame(rows)


def short_long_difference(bands: pd.DataFrame) -> pd.DataFrame:
    if not {"short", "long"}.issubset(set(bands["class"])):
        return pd.DataFrame()

    numeric_columns = [
        "timepoint_mean",
        "timepoint_std",
        "per_grb_mean_avg",
        "per_grb_peak_avg",
        "per_grb_sum_avg",
        "zero_fraction",
    ]
    short = bands[bands["class"] == "short"].set_index("band")
    long = bands[bands["class"] == "long"].set_index("band")
    rows = []
    for band in short.index.intersection(long.index):
        row = {"band": band}
        for column in numeric_columns:
            short_value = float(short.loc[band, column])
            long_value = float(long.loc[band, column])
            row[f"{column}_long_minus_short"] = long_value - short_value
            row[f"{column}_long_div_short"] = (
                long_value / short_value if not np.isclose(short_value, 0.0) else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows)

In [78]:
def full_grb_dataframe(
    dataset: GRBHDF5Dataset,
    x: np.ndarray,
    labels: np.ndarray,
    label: int,
) -> tuple[str, float, pd.DataFrame]:
    indices = np.where(labels == label)[0]
    if indices.size == 0:
        raise ValueError(f"No {class_label(label)} GRB found")

    idx = int(indices[0])
    grb = pd.DataFrame(x[idx], columns=dataset.channel_columns)
    grb.insert(0, "time_index", np.arange(len(grb)))
    return dataset.names[idx], dataset.t90[idx], grb

## Load dataset

In [79]:
dataset = GRBHDF5Dataset(H5_FILE)
x, labels, t90 = as_numpy(dataset)

print(f"HDF5 file: {H5_FILE}")
print(f"Loaded GRBs: {len(dataset)}")
print(f"Input shape: {tuple(x.shape)}")
print(f"Channels: {', '.join(dataset.channel_columns)}")
print(f"Label rule: {dataset.label_rule}")

HDF5 file: /Users/daniele/Desktop/esami mancanti/grb-thesis/Spectral-Evolution-of-a-GRB/data/processed/classipygrb/swift.hd5
Loaded GRBs: 268
Input shape: (268, 313, 4)
Channels: 15-25keV, 25-50keV, 50-100keV, 100-350keV
Label rule: 0 short, 1 long


## Class distribution

In [80]:
class_distribution(labels)

,label,class,count,percent
0,0,short,134,50.0000
1,1,long,134,50.0000


## How time, channels, and T90 fit together

- `t90` is one number for the whole GRB. It is the duration used to label the burst as short or long.
- `time_index` is only the row position inside the processed light curve array. It is not T90 and it is not necessarily seconds.
- Each channel/band column is one energy band value at that processed time position.
- The HDF5 file stores a fixed-size tensor, so every GRB has the same number of time rows. Short bursts often have many zero rows because the useful signal is short compared with the fixed length.

In [81]:
print(f"X shape: {x.shape}")
print("Meaning: (number_of_grbs, processed_time_rows, channels)")
print(f"Number of GRBs: {x.shape[0]}")
print(f"Processed time rows per GRB: {x.shape[1]}")
print(f"Number of channels/bands: {x.shape[2]}")
print(f"Channels: {dataset.channel_columns}")

pd.DataFrame(
    {
        "object": ["t90", "time_index", "channel value"],
        "where_it_lives": ["dataset.t90, one value per GRB", "row index inside X", "X[grb_index, time_index, channel_index]"],
        "meaning": [
            "duration of the burst; used for short/long label",
            "processed row number; not automatically seconds",
            "light-curve value for one band at one processed time row",
        ],
    }
)

X shape: (268, 313, 4)
Meaning: (number_of_grbs, processed_time_rows, channels)
Number of GRBs: 268
Processed time rows per GRB: 313
Number of channels/bands: 4
Channels: ['15-25keV', '25-50keV', '50-100keV', '100-350keV']


,object,where_it_lives,meaning
0,t90,"dataset.t90, one value per GRB",duration of the burst; used for short/long label
1,time_index,row index inside X,processed row number; not automatically seconds
2,channel value,"X[grb_index, time_index, channel_index]",light-curve value for one band at one processe...


In [82]:
def nonzero_row_summary(grb_df: pd.DataFrame) -> dict[str, int | None]:
    band_cols = [column for column in grb_df.columns if column != "time_index"]
    nonzero_rows = (grb_df[band_cols] != 0).any(axis=1)

    if not nonzero_rows.any():
        first_nonzero = None
        last_nonzero = None
    else:
        nonzero_indices = grb_df.loc[nonzero_rows, "time_index"]
        first_nonzero = int(nonzero_indices.iloc[0])
        last_nonzero = int(nonzero_indices.iloc[-1])

    return {
        "total_rows": int(len(grb_df)),
        "rows_with_any_signal": int(nonzero_rows.sum()),
        "all_zero_rows": int((~nonzero_rows).sum()),
        "first_nonzero_time_index": first_nonzero,
        "last_nonzero_time_index": last_nonzero,
    }


short_name, short_t90, short_grb = full_grb_dataframe(dataset, x, labels, label=0)
long_name, long_t90, long_grb = full_grb_dataframe(dataset, x, labels, label=1)

pd.DataFrame(
    [
        {"class": "short", "name": short_name, "t90": short_t90, **nonzero_row_summary(short_grb)},
        {"class": "long", "name": long_name, "t90": long_t90, **nonzero_row_summary(long_grb)},
    ]
)

,class,name,t90,total_rows,rows_with_any_signal,all_zero_rows,first_nonzero_time_index,last_nonzero_time_index
0,short,GRB050202,0.1120,313,3,310,0,2
1,long,GRB041217,5.6680,313,177,136,0,176


In [83]:
def compact_grb_preview(grb_df: pd.DataFrame, edge_rows: int = 8) -> pd.DataFrame:
    if len(grb_df) <= edge_rows * 2:
        return grb_df

    separator = pd.DataFrame(
        [{column: "..." for column in grb_df.columns}],
        columns=grb_df.columns,
    )
    return pd.concat([grb_df.head(edge_rows), separator, grb_df.tail(edge_rows)], ignore_index=True)


print(f"Short GRB preview: {short_name}, t90={short_t90:.4g}")
display(compact_grb_preview(short_grb))

print(f"Long GRB preview: {long_name}, t90={long_t90:.4g}")
display(compact_grb_preview(long_grb))

Short GRB preview: GRB050202, t90=0.112


,time_index,15-25keV,25-50keV,50-100keV,100-350keV
0,0,-0.0204,0.0219,0.0180,-0.0119
1,1,0.0998,0.1877,0.1782,0.0274
2,2,0.0416,0.1169,0.0870,0.0334
3,3,0.0000,0.0000,0.0000,0.0000
4,4,0.0000,0.0000,0.0000,0.0000
5,5,0.0000,0.0000,0.0000,0.0000
6,6,0.0000,0.0000,0.0000,0.0000
7,7,0.0000,0.0000,0.0000,0.0000
8,...,...,...,...,...
9,305,0.0000,0.0000,0.0000,0.0000


Long GRB preview: GRB041217, t90=5.668


,time_index,15-25keV,25-50keV,50-100keV,100-350keV
0,0,-0.0350,0.0133,-0.0239,-0.0900
1,1,0.0201,0.0093,0.0516,-0.0400
2,2,-0.0378,0.0997,0.1070,0.0871
3,3,0.0406,0.0368,-0.0109,0.0262
4,4,0.0477,0.0592,0.0537,0.0237
5,5,-0.1250,0.1725,-0.1297,0.0208
6,6,0.0488,0.0504,0.0733,0.0359
7,7,-0.1107,0.0795,0.1080,-0.0065
8,...,...,...,...,...
9,305,0.0000,0.0000,0.0000,0.0000


## Are the zeros saved in HDF5 or created by ClassiPy?

The zeros you see here are already inside the HDF5 `X` dataset loaded from disk. They are not created by the notebook display.

ClassiPyGRB provides the raw Swift light-curve data. In this project, before saving the HDF5 cache, the light curves were converted to fixed-length arrays. During that preprocessing, short or incomplete curves can be padded with zeros, and missing/non-finite values can also become zeros. So the zeros are saved in HDF5, but they mainly come from the project preprocessing step rather than being a new effect of this notebook.

In [84]:
zero_report = []
for label in sorted(np.unique(labels)):
    class_x = x[labels == label]
    zero_report.append(
        {
            "class": class_label(int(label)),
            "values_in_h5_X": int(class_x.size),
            "zeros_in_h5_X": int((class_x == 0).sum()),
            "zero_fraction": float((class_x == 0).mean()),
        }
    )

pd.DataFrame(zero_report)

,class,values_in_h5_X,zeros_in_h5_X,zero_fraction
0,short,167768,158885,0.9471
1,long,167768,86838,0.5176


## Full short GRB example

In [85]:
short_name, short_t90, short_grb = full_grb_dataframe(dataset, x, labels, label=0)
print(f"Short GRB: name={short_name}, label=0, t90={short_t90:.4g}")

with pd.option_context("display.max_rows", None):
    display(short_grb)

Short GRB: name=GRB050202, label=0, t90=0.112


,time_index,15-25keV,25-50keV,50-100keV,100-350keV
0,0,-0.0204,0.0219,0.0180,-0.0119
1,1,0.0998,0.1877,0.1782,0.0274
2,2,0.0416,0.1169,0.0870,0.0334
3,3,0.0000,0.0000,0.0000,0.0000
4,4,0.0000,0.0000,0.0000,0.0000
5,5,0.0000,0.0000,0.0000,0.0000
6,6,0.0000,0.0000,0.0000,0.0000
7,7,0.0000,0.0000,0.0000,0.0000
8,8,0.0000,0.0000,0.0000,0.0000
9,9,0.0000,0.0000,0.0000,0.0000


## Full long GRB example

In [86]:
long_name, long_t90, long_grb = full_grb_dataframe(dataset, x, labels, label=1)
print(f"Long GRB: name={long_name}, label=1, t90={long_t90:.4g}")

with pd.option_context("display.max_rows", None):
    display(long_grb)

Long GRB: name=GRB041217, label=1, t90=5.668


,time_index,15-25keV,25-50keV,50-100keV,100-350keV
0,0,-0.0350,0.0133,-0.0239,-0.0900
1,1,0.0201,0.0093,0.0516,-0.0400
2,2,-0.0378,0.0997,0.1070,0.0871
3,3,0.0406,0.0368,-0.0109,0.0262
4,4,0.0477,0.0592,0.0537,0.0237
5,5,-0.1250,0.1725,-0.1297,0.0208
6,6,0.0488,0.0504,0.0733,0.0359
7,7,-0.1107,0.0795,0.1080,-0.0065
8,8,0.0203,0.0150,0.0604,0.0586
9,9,-0.0399,0.0270,0.1063,0.0050


## T90 summary by class

In [87]:
t90_summary(labels, t90)

,class,mean,std,median,min,q25,q75,max
0,short,0.5541,0.4839,0.3840,0.0120,0.1800,0.8250,2.0000
1,long,5.9119,2.2997,6.0320,2.0920,3.9270,7.7680,10.0000


## Overall light-curve summary by class

In [88]:
overall_signal_summary(x, labels)

,class,all_values_mean,all_values_std,per_grb_mean,per_grb_peak_mean,per_grb_sum_mean,zero_fraction
0,short,0.0023,0.0324,0.0023,0.3615,2.8410,0.9471
1,long,0.0188,0.1421,0.0188,0.4080,23.5707,0.5176


## Average values across bands by class

In [89]:
bands = band_summary(x, labels, dataset.channel_columns)
bands

,class,band,timepoint_mean,timepoint_std,per_grb_mean_avg,per_grb_peak_avg,per_grb_sum_avg,zero_fraction
0,short,15-25keV,0.0019,0.0263,0.0019,0.1932,0.6033,0.9470
1,short,25-50keV,0.0031,0.0378,0.0031,0.3029,0.9818,0.9470
2,short,50-100keV,0.0030,0.0397,0.0030,0.3120,0.9399,0.9470
3,short,100-350keV,0.0010,0.0221,0.0010,0.1679,0.3160,0.9473
4,long,15-25keV,0.0213,0.1588,0.0213,0.2992,6.6691,0.5173
5,long,25-50keV,0.0284,0.1886,0.0284,0.3691,8.8896,0.5174
6,long,50-100keV,0.0206,0.1316,0.0206,0.3221,6.4361,0.5176
7,long,100-350keV,0.0050,0.0482,0.0050,0.1829,1.5760,0.5181


## Long versus short differences by band

In [90]:
short_long_difference(bands)

,band,timepoint_mean_long_minus_short,timepoint_mean_long_div_short,timepoint_std_long_minus_short,timepoint_std_long_div_short,per_grb_mean_avg_long_minus_short,per_grb_mean_avg_long_div_short,per_grb_peak_avg_long_minus_short,per_grb_peak_avg_long_div_short,per_grb_sum_avg_long_minus_short,per_grb_sum_avg_long_div_short,zero_fraction_long_minus_short,zero_fraction_long_div_short
0,15-25keV,0.0194,11.0546,0.1325,6.0347,0.0194,11.0546,0.1060,1.5486,6.0658,11.0546,-0.4297,0.5463
1,25-50keV,0.0253,9.0546,0.1508,4.9912,0.0253,9.0546,0.0662,1.2185,7.9078,9.0546,-0.4295,0.5464
2,50-100keV,0.0176,6.8475,0.0918,3.3111,0.0176,6.8475,0.0101,1.0323,5.4962,6.8475,-0.4294,0.5466
3,100-350keV,0.0040,4.9866,0.0260,2.1770,0.0040,4.9866,0.0150,1.0893,1.2600,4.9866,-0.4292,0.5469
